# POI coordinate check
Fetches all POIs from the DB and flags any outside Barcelona's bounding box.

In [1]:
import sys
from pathlib import Path

import folium
import pandas as pd

_db_dir = next(
    (p / "src" / "db_upload")
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "src" / "db_upload" / "_db.py").exists()
)
sys.path.insert(0, str(_db_dir))
from _db import connect, fetch_all

In [2]:
conn = connect()
rows = fetch_all(conn, "SELECT register_id, name, category, lat, lon FROM pois ORDER BY name;")
conn.close()

df = pd.DataFrame(rows, columns=["register_id", "name", "category", "lat", "lon"])
print(f"{len(df)} POIs loaded from DB")

878 POIs loaded from DB


In [3]:
BCN_LAT = (41.32, 41.50)
BCN_LON = (2.05, 2.23)

outliers = df[
    ~df["lat"].between(*BCN_LAT) | ~df["lon"].between(*BCN_LON)
]

print(f"Outliers outside Barcelona bbox: {len(outliers)}")
outliers[["register_id", "name", "category", "lat", "lon"]]

Outliers outside Barcelona bbox: 0


,register_id,name,category,lat,lon


In [4]:
# Colour outliers red, valid POIs blue
m = folium.Map(location=[41.39, 2.15], zoom_start=12)

# Barcelona bounding box rectangle
folium.Rectangle(
    bounds=[[BCN_LAT[0], BCN_LON[0]], [BCN_LAT[1], BCN_LON[1]]],
    color="orange", fill=False, weight=2, dash_array="6",
    tooltip="Barcelona bbox",
).add_to(m)

for _, row in df.iterrows():
    is_outlier = not (BCN_LAT[0] <= row["lat"] <= BCN_LAT[1] and BCN_LON[0] <= row["lon"] <= BCN_LON[1])
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=6 if is_outlier else 4,
        color="red" if is_outlier else "steelblue",
        fill=True,
        fill_opacity=0.9 if is_outlier else 0.6,
        popup=folium.Popup(
            f"<b>{row['name']}</b><br>{row['category']}<br>lat={row['lat']:.5f}, lon={row['lon']:.5f}",
            max_width=300,
        ),
        tooltip=row["name"] if is_outlier else None,
    ).add_to(m)

m

## Fix outliers
Run the cell below to delete outlier POIs from the DB (requires confirmation).

In [ ]:
if len(outliers) == 0:
    print("No outliers to remove.")
else:
    print("Outliers that would be deleted:")
    print(outliers[["register_id", "name", "lat", "lon"]].to_string(index=False))
    confirm = input("\nType YES to delete from DB: ")
    if confirm.strip() == "YES":
        import psycopg2.extras
        ids = outliers["register_id"].tolist()
        conn = connect()
        with conn.cursor() as cur:
            cur.execute("DELETE FROM pois WHERE register_id = ANY(%s)", (ids,))
            print(f"Deleted {cur.rowcount} rows.")
        conn.commit()
        conn.close()
    else:
        print("Aborted.")